# Agents as Python Control Flow
Build a bounded tool workflow with state, traces, guardrails, and human confirmation.

## 1. Define ordinary tools

In [ ]:
def get_market_price(symbol):
    return {"symbol": symbol, "price": 228.80}

def get_portfolio():
    return {"cash": 5_000, "holdings": ["SPY"]}

def get_risk(symbol):
    return {"symbol": symbol, "risk": "MODERATE"}

## 2. Register tools

In [ ]:
TOOLS = {
    "get_market_price": get_market_price,
    "get_portfolio": get_portfolio,
    "get_risk": get_risk,
}
print(list(TOOLS))

## 3. Select deterministically

In [ ]:
def select_tool(question):
    if "price" in question.lower():
        return "get_market_price", {"symbol": "SPY"}
    if "risk" in question.lower():
        return "get_risk", {"symbol": "SPY"}
    return "get_portfolio", {}

selection = select_tool("What is the SPY price?")
print(selection)

A model could choose a tool here, but this teaching workflow uses a deterministic Python condition.

## 4. Dispatch by name

In [ ]:
tool_name, arguments = selection
observation = TOOLS[tool_name](**arguments)
print(observation)

## 5. Store state

In [ ]:
state = {"question": "What is the SPY price?", "step": 0, "finished": False}
state["observation"] = observation
print(state)

## 6. Use a bounded loop and trace

In [ ]:
trace = []
for step in range(3):
    name, args = select_tool(state["question"])
    result = TOOLS[name](**args)
    trace.append({"tool": name, "arguments": args, "result": result})
    state["step"] = step + 1
    state["finished"] = True
    break
print(trace)

## 7. Describe a proposed action

In [ ]:
from pydantic import BaseModel, PositiveFloat

class ProposedAction(BaseModel):
    symbol: str
    amount: PositiveFloat

## 8. Add the cash guardrail

In [ ]:
MINIMUM_CASH = 2_000

def check_guardrail(action, cash):
    if cash - action.amount < MINIMUM_CASH:
        return "REJECTED_MINIMUM_CASH"
    return "AWAITING_HUMAN_CONFIRMATION"

## 9. Stop for a human

In [ ]:
safe_action = ProposedAction(symbol="SPY", amount=2_000)
status = check_guardrail(safe_action, cash=5_000)
state["status"] = status
print(status)
assert status == "AWAITING_HUMAN_CONFIRMATION"

## 10. Deliberate failure: unsafe action

In [ ]:
unsafe_action = ProposedAction(symbol="SPY", amount=3_500)
unsafe_status = check_guardrail(unsafe_action, cash=5_000)
print(unsafe_status)

## 11. Record the decision

In [ ]:
trace.append({
    "tool": "check_guardrail",
    "arguments": unsafe_action.model_dump(),
    "result": unsafe_status,
})
print(trace[-1])

## 12. Final workflow state

In [ ]:
state["finished"] = status == "AWAITING_HUMAN_CONFIRMATION"
print(state)
print("No trade was executed.")

## Takeaways
- Tools are ordinary functions stored by name.
- State and bounded loops make control flow visible.
- Guardrails reject unsafe actions and humans confirm safe proposals.